# 🔬 CSV seul + Spectres propres en Train — Run B (point 2, Dr. Sarun)
## Importance des données .npy — sans elles, mais avec les CSV clean en renfort
### CMKL University · Stage 2026

---

**Question posée (Dr. Sarun)** : à quel point les données synthétiques .npy
sont-elles importantes ? Peut-on s'en passer si on ajoute les spectres CSV
**propres** comme exemples de classification supplémentaires dans le train ?

**Comparaison à établir** :
```
Run A (déjà fait, CSV_MultiTask.ipynb) :
  CSV bruités SEULS en train (propres utilisés UNIQUEMENT comme cible proxy)
  → 94.21% (config CSV optimale de l'époque, P=64/S=48, beta=0.2)

Run B (CE NOTEBOOK) :
  CSV bruités + CSV propres EN PLUS comme exemples de classification
  (les propres gardent aussi leur rôle de cible proxy pour le denoising)
  → résultat à déterminer
```

**Ce qui change concrètement** : dans le run précédent, un spectre CSV propre
ne servait qu'à calculer la moyenne de classe (cible de denoising). Ici, CHAQUE
spectre propre devient AUSSI un exemple d'entraînement à part entière pour la
classification — avec pour cible de denoising lui-même (puisqu'il est déjà propre).

**Aucune donnée .npy n'est chargée dans ce notebook** — c'est exactement le point
de la question de Dr. Sarun : peut-on se passer du .npy synthétique ?


---
## ⚙️ Section 0 — Imports & Configuration


In [1]:
import subprocess, sys
def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
for pkg in ['scikit-learn', 'seaborn']:
    try: __import__(pkg.replace('-','_'))
    except ImportError: install(pkg)
print('✓ Packages prêts')

✓ Packages prêts


In [2]:
import os, glob, re, io, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, OneCycleLR

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HOME   = os.path.expanduser('~')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

Device : cuda
GPU    : NVIDIA A100-SXM4-40GB


In [3]:
from torch.utils.tensorboard import SummaryWriter

LOG_DIR = os.path.join(HOME, 'runs', 'patchtst_csv_with_clean')
os.makedirs(LOG_DIR, exist_ok=True)
writer  = SummaryWriter(LOG_DIR)
print(f'✓ TensorBoard logs → {LOG_DIR}')

def safe_log(writer, *args, method='add_scalar', **kwargs):
    try: getattr(writer, method)(*args, **kwargs)
    except Exception: pass

✓ TensorBoard logs → /home/glider/runs/patchtst_csv_with_clean


In [4]:
# ════════════════════════════════════════════════════════════════════
# CONFIG — même architecture/config finale que le mixte, pour comparaison
# rigoureuse avec le Run A (si tu relances le Run A avec cette même config)
# ════════════════════════════════════════════════════════════════════
CFG = {
    'L'         : 6700,
    'WN_MIN'    : 650,
    'WN_MAX'    : 4000,
    'WN_STEP'   : 0.5,

    'patch_size' : 320,
    'stride'     : 256,

    'd_model'   : 256,
    'n_heads'   : 16,
    'n_layers'  : 5,
    'd_ff'      : 512,
    'dropout'   : 0.1,

    'mask_ratio'  : 0.40,
    'ssl_epochs'  : 100,
    'ssl_lr'      : 1e-3,
    'ssl_batch'   : 64,

    'alpha'        : 1.0,
    'beta'         : 0.5,
    'probe_epochs' : 80,
    'probe_lr'     : 1e-3,
    'ft_epochs'    : 80,
    'ft_lr'        : 1e-5,
    'batch_size'   : 32,

    'ssl_path'   : os.path.join(HOME, 'models', 'ssl_backbone_csv_withclean.pth'),
    'probe_path' : os.path.join(HOME, 'models', 'probe_model_csv_withclean.pth'),
    'final_path' : os.path.join(HOME, 'models', 'final_model_csv_withclean.pth'),
}
os.makedirs(os.path.join(HOME, 'models'), exist_ok=True)

WN_GRID   = np.arange(CFG['WN_MIN'], CFG['WN_MAX'], CFG['WN_STEP'])
CFG['L']  = len(WN_GRID)
L = CFG['L']
N_PATCHES = (L - CFG['patch_size']) // CFG['stride'] + 2
print(f'L = {L}, N_PATCHES = {N_PATCHES}')
assert CFG['d_model'] % CFG['n_heads'] == 0

L = 6700, N_PATCHES = 26


In [5]:
ASSUMED_CLASSES = [
    'ABS', 'ACRYLIC', 'CELLULOSE', 'CHITOSAN', 'ENR', 'EPDM', 'EVA', 'HDPE',
    'LDPE', 'NYLON', 'PBAT', 'PBS', 'PC', 'PEEK', 'PEI', 'PET',
    'PF THERMOPLASTIC', 'PF THERMOSET', 'PHB', 'PLA', 'PMMA', 'POM', 'PP',
    'PS', 'PTFE', 'PU', 'PVA', 'PVC', 'PVDF', 'SAN',
]
N_CLASSES = len(ASSUMED_CLASSES)
CFG['N_CLASSES'] = N_CLASSES
le = LabelEncoder()
le.fit(ASSUMED_CLASSES)
print(f'{N_CLASSES} classes')

30 classes


---
## 📁 Section 1 — Chargement CSV (PAS de .npy dans ce notebook)


In [6]:
CSV_ROOT = os.path.join(HOME, 'data', '2026-FirstDataSet', '2026 - Complete FTIR Dataset')

PATHS_CSV = {
    '2023_base' : os.path.join(CSV_ROOT, '2023 Dataset - 22 MP Types with 10 Clean and 60 Noisy'),
    '2025_ext'  : os.path.join(CSV_ROOT, '2025 Dataset 1 - Same 22 MP Types - Add 40 Spectra'),
    '2025_new'  : os.path.join(CSV_ROOT, '2025 Dataset 2 - New 9 MP Types - 50 Clean and 100 Noisy'),
}
for name, path in PATHS_CSV.items():
    status = '✓' if os.path.exists(path) else '✗ INTROUVABLE'
    print(f'  {status}  {name}')

  ✓  2023_base
  ✓  2025_ext
  ✓  2025_new


In [7]:
EXCLUDE_FILES = {'ref.csv', 'reference.csv', 'background.csv', 'bg.csv'}

def is_noisy_csv(filepath):
    p = str(filepath).lower()
    if any(k in p for k in ['noisy', '_sd', '-sd', 'sd_']): return True
    if any(k in p for k in ['clean', '_rm', '-rm', 'rm_']): return False
    return False

def extract_label_csv(filepath):
    name = Path(filepath).stem.upper()
    for pattern in ['_SD_', '_RM_', '_NOISY', '_CLEAN', 'PARTICLE', '-NOISY',
                    '-CLEAN', '_50', '_60', '_40', '_100', '_10', '_30',
                    ' SPECTRUMS', ' SPECTUMS', 'ADD_40', '-ADD_40']:
        name = name.replace(pattern, ' ')
    name = re.sub(r'\d+', '', name)
    name = re.sub(r'\bNEW\b|\bJAN\b|\bX\b', '', name)
    name = ' '.join(name.replace('_', ' ').replace('-', ' ').split())
    MAPPING = {
        'NYLON PARTICLE' : 'NYLON', 'PTEE' : 'PTFE', 'PTFE' : 'PTFE',
        'PF THERMOPLASTIC CLEAN' : 'PF THERMOPLASTIC',
        'PF THERMOSET CLEAN'     : 'PF THERMOSET',
    }
    if name in MAPPING: return MAPPING[name]
    if name in ASSUMED_CLASSES: return name
    for cls in ASSUMED_CLASSES:
        if cls in name or name in cls: return cls
    return None

def read_csv_multispectra(filepath, sep=','):
    try:
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines()
        start_idx = 0
        for i, line in enumerate(lines):
            parts = line.strip().split(sep)
            if len(parts) >= 2:
                try:
                    float(parts[0].replace(',', '.'))
                    start_idx = i; break
                except ValueError: continue
        valid = ''.join(lines[start_idx:])
        headers = lines[start_idx-1].strip().split(sep) if start_idx > 0 else []
        try:
            df = pd.read_csv(io.StringIO(valid), sep=sep, header=None, decimal=',')
        except Exception:
            df = pd.read_csv(io.StringIO(valid), sep=sep, header=None, decimal='.')
        cols = []
        for ci, cn in enumerate(df.columns):
            h = headers[ci].upper() if ci < len(headers) else ''
            v = str(df[cn].iloc[0]).upper()
            if any(k in h for k in ['AIR','BACKGROUND','BG']): continue
            if any(k in v for k in ['AIR','BACKGROUND','BG']): continue
            cols.append(cn)
        df = df[cols].apply(pd.to_numeric, errors='coerce')
        df = df.dropna(subset=[df.columns[0]])
        if len(df) < 100: return None
        wn    = df.iloc[:, 0].values.astype(float)
        order = np.argsort(wn); wn = wn[order]
        spectra = []
        for c in range(1, df.shape[1]):
            ab = df.iloc[order, c].values.astype(float)
            if np.isnan(ab).all() or ab.std() < 1e-10: continue
            nans = np.isnan(ab)
            if nans.any():
                ab[nans] = np.interp(np.where(nans)[0], np.where(~nans)[0], ab[~nans])
            spectra.append(ab.astype(np.float32))
        return (wn, spectra) if spectra else None
    except Exception: return None

print('✓ Fonctions de lecture définies')

✓ Fonctions de lecture définies


In [8]:
print('Chargement des CSV (propres + bruités)...')
csv_records = []
for src_name, folder in PATHS_CSV.items():
    if not os.path.exists(folder): continue
    files = glob.glob(os.path.join(folder, '**/*.csv'), recursive=True)
    n_ok = 0
    for fp in files:
        if Path(fp).name.lower() in EXCLUDE_FILES: continue
        label = extract_label_csv(fp)
        if label is None: continue
        result = read_csv_multispectra(fp)
        if result is None: continue
        wn, spectra_list = result
        noisy = is_noisy_csv(fp)
        for sp in spectra_list:
            sp_interp = np.interp(WN_GRID, wn, sp).astype(np.float32)
            csv_records.append({'label': label, 'is_noisy': noisy, 'spectrum': sp_interp})
            n_ok += 1
    print(f'  ✓ {src_name:12s} : {n_ok:4d} spectres')

df_csv_all = pd.DataFrame(csv_records)
df_csv_all['label_enc'] = le.transform(df_csv_all['label'])
df_csv_clean = df_csv_all[~df_csv_all['is_noisy']].reset_index(drop=True)
df_csv_noisy = df_csv_all[ df_csv_all['is_noisy']].reset_index(drop=True)

print(f'\n  Total CSV   : {len(df_csv_all)} spectres')
print(f'  Propres (RM): {len(df_csv_clean)}')
print(f'  Bruités (SD): {len(df_csv_noisy)}')

Chargement des CSV (propres + bruités)...
  ✓ 2023_base    : 1518 spectres
  ✓ 2025_ext     :  881 spectres
  ✓ 2025_new     : 1145 spectres

  Total CSV   : 3544 spectres
  Propres (RM): 881
  Bruités (SD): 2663


In [9]:
# ── Cibles de denoising proxy par classe (pour les spectres BRUITÉS) ──────
csv_clean_reference = {}
for cls_idx in range(N_CLASSES):
    subset = df_csv_clean[df_csv_clean['label_enc'] == cls_idx]['spectrum']
    if len(subset) > 0:
        csv_clean_reference[cls_idx] = np.mean(np.stack(subset.values), axis=0).astype(np.float32)
csv_global_clean_mean = (np.mean(np.stack(df_csv_clean['spectrum'].values), axis=0).astype(np.float32)
                         if len(df_csv_clean) > 0 else np.zeros(L, dtype=np.float32))
print(f'Référence propre disponible pour {len(csv_clean_reference)}/{N_CLASSES} classes')

Référence propre disponible pour 21/30 classes


### 🆕 Split — les spectres PROPRES sont ajoutés au Train (le cœur du Run B)

```
Run A (déjà fait) :
  Train = bruités_train  (propres SEULEMENT utilisés pour calculer la cible proxy)

Run B (ici) :
  Train = bruités_train + TOUS les propres
          (propres = exemples de classification À PART ENTIÈRE,
           avec pour cible de denoising : eux-mêmes, car déjà propres)
```


In [10]:
# ── Split des BRUITÉS : 70/15/15 comme d'habitude (Val/Test inchangés) ────
csv_noisy_counts = Counter(df_csv_noisy['label_enc'])
csv_singleton = {k for k, v in csv_noisy_counts.items() if v < 3}
df_csv_multi  = df_csv_noisy[~df_csv_noisy['label_enc'].isin(csv_singleton)]
df_csv_single = df_csv_noisy[ df_csv_noisy['label_enc'].isin(csv_singleton)]

idx_tr_csv, idx_valtest_csv = train_test_split(
    range(len(df_csv_multi)), test_size=0.3,
    random_state=SEED, stratify=df_csv_multi['label_enc'])
idx_val_csv, idx_test_csv = train_test_split(idx_valtest_csv, test_size=0.5, random_state=SEED)

df_csv_noisy_train = pd.concat([df_csv_multi.iloc[idx_tr_csv], df_csv_single]).reset_index(drop=True)
df_csv_val  = df_csv_multi.iloc[idx_val_csv].reset_index(drop=True)
df_csv_test = df_csv_multi.iloc[idx_test_csv].reset_index(drop=True)

print(f'Bruités Train : {len(df_csv_noisy_train)}')
print(f'Val  (bruités uniquement, FIXE) : {len(df_csv_val)}')
print(f'Test (bruités uniquement, FIXE) : {len(df_csv_test)}')
print(f'Propres disponibles pour AJOUT au train : {len(df_csv_clean)}')

Bruités Train : 1864
Val  (bruités uniquement, FIXE) : 399
Test (bruités uniquement, FIXE) : 400
Propres disponibles pour AJOUT au train : 881


In [11]:
class CSVWithCleanDataset(Dataset):
    """
    Combine bruités (cible proxy = moyenne classe) et propres
    (cible = eux-mêmes, déjà propres) comme exemples de classification.
    """
    def __init__(self, noisy_df, clean_df, clean_ref, global_mean):
        self.noisy_spectra = np.stack(noisy_df['spectrum'].values).astype(np.float32) \
                              if len(noisy_df) > 0 else np.zeros((0, len(global_mean)), np.float32)
        self.noisy_labels  = noisy_df['label_enc'].values.astype(np.int64) \
                              if len(noisy_df) > 0 else np.zeros(0, np.int64)
        self.n_noisy = len(self.noisy_labels)

        self.clean_spectra = np.stack(clean_df['spectrum'].values).astype(np.float32) \
                              if len(clean_df) > 0 else np.zeros((0, len(global_mean)), np.float32)
        self.clean_labels  = clean_df['label_enc'].values.astype(np.int64) \
                              if len(clean_df) > 0 else np.zeros(0, np.int64)
        self.n_clean = len(self.clean_labels)

        self.clean_ref   = clean_ref
        self.global_mean = global_mean

    def __len__(self):
        return self.n_noisy + self.n_clean

    def __getitem__(self, idx):
        if idx < self.n_noisy:
            x = torch.tensor(self.noisy_spectra[idx], dtype=torch.float32)
            y = torch.tensor(self.noisy_labels[idx], dtype=torch.long)
            ref = self.clean_ref.get(int(self.noisy_labels[idx]), self.global_mean)
            x_clean = torch.tensor(ref, dtype=torch.float32)
        else:
            i = idx - self.n_noisy
            x = torch.tensor(self.clean_spectra[i], dtype=torch.float32)
            y = torch.tensor(self.clean_labels[i], dtype=torch.long)
            x_clean = x.clone()   # déjà propre → cible = lui-même

        mu, sigma = x.mean(), x.std() + 1e-8
        x       = (x - mu) / sigma
        x_clean = (x_clean - mu) / sigma
        return x, x_clean, y


class SimpleMultiTaskDataset(Dataset):
    def __init__(self, noisy, clean, labels):
        self.noisy  = noisy.astype(np.float32)
        self.clean  = clean.astype(np.float32)
        self.labels = labels.astype(np.int64)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        x       = torch.tensor(self.noisy[idx], dtype=torch.float32)
        x_clean = torch.tensor(self.clean[idx], dtype=torch.float32)
        y       = torch.tensor(self.labels[idx], dtype=torch.long)
        mu, sigma = x.mean(), x.std() + 1e-8
        x       = (x - mu) / sigma
        x_clean = (x_clean - mu) / sigma
        return x, x_clean, y


train_dataset = CSVWithCleanDataset(df_csv_noisy_train, df_csv_clean,
                                     csv_clean_reference, csv_global_clean_mean)

csv_val_clean_targets = np.stack([csv_clean_reference.get(int(l), csv_global_clean_mean)
                                  for l in df_csv_val['label_enc'].values])
val_dataset = SimpleMultiTaskDataset(np.stack(df_csv_val['spectrum'].values),
                                     csv_val_clean_targets, df_csv_val['label_enc'].values)

csv_test_clean_targets = np.stack([csv_clean_reference.get(int(l), csv_global_clean_mean)
                                   for l in df_csv_test['label_enc'].values])
test_dataset = SimpleMultiTaskDataset(np.stack(df_csv_test['spectrum'].values),
                                      csv_test_clean_targets, df_csv_test['label_enc'].values)

print(f'✓ train_dataset : {len(train_dataset)} (bruités={train_dataset.n_noisy}, propres AJOUTÉS={train_dataset.n_clean})')
print(f'✓ val_dataset   : {len(val_dataset)} (bruités uniquement)')
print(f'✓ test_dataset  : {len(test_dataset)} (bruités uniquement, FIXE)')

✓ train_dataset : 2745 (bruités=1864, propres AJOUTÉS=881)
✓ val_dataset   : 399 (bruités uniquement)
✓ test_dataset  : 400 (bruités uniquement, FIXE)


In [12]:
# ── Sampler pondéré par classe (déséquilibre naturel CSV) ─────────────────
all_labels_for_weight = np.concatenate([train_dataset.noisy_labels, train_dataset.clean_labels])
class_counts = np.bincount(all_labels_for_weight, minlength=N_CLASSES)
sample_weights = 1.0 / (class_counts[all_labels_for_weight] + 1e-8)
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.float32),
    num_samples=len(train_dataset), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=CFG['batch_size'], sampler=sampler, num_workers=0)
val_loader   = DataLoader(val_dataset, batch_size=CFG['batch_size'], shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset, batch_size=CFG['batch_size'], shuffle=False, num_workers=0)

xb, xb_clean, yb = next(iter(train_loader))
print(f'Train batch : {xb.shape}')
print('✓ DataLoaders prêts')

Train batch : torch.Size([32, 6700])
✓ DataLoaders prêts


In [13]:
class SSLPoolDataset(Dataset):
    def __init__(self, arrays, max_n=None, seed=SEED):
        rng = np.random.RandomState(seed)
        if max_n is not None:
            n_per_array = max_n // len(arrays)
            sampled = []
            for arr in arrays:
                if len(arr) > n_per_array:
                    idx = rng.choice(len(arr), size=n_per_array, replace=False)
                    sampled.append(arr[idx].astype(np.float32))
                else:
                    sampled.append(arr.astype(np.float32))
            self.spectra = np.concatenate(sampled, axis=0)
        else:
            self.spectra = np.concatenate(arrays, axis=0).astype(np.float32)
    def __len__(self): return len(self.spectra)
    def __getitem__(self, idx):
        x = torch.tensor(self.spectra[idx], dtype=torch.float32)
        x = (x - x.mean()) / (x.std() + 1e-8)
        return x

csv_all_spectra = np.stack(df_csv_all['spectrum'].values)
ssl_dataset = SSLPoolDataset(arrays=[csv_all_spectra], max_n=None)
print(f'✓ SSL pool (CSV uniquement, PAS de .npy) : {len(ssl_dataset)} spectres')

✓ SSL pool (CSV uniquement, PAS de .npy) : 3544 spectres


---
## 🏛️ Section 2 — Architecture Conformer (identique à la config finale)


In [14]:
class PatchEmbedding(nn.Module):
    def __init__(self, L, patch_size, stride, d_model):
        super().__init__()
        self.P, self.S, self.D = patch_size, stride, d_model
        self.N = (L - patch_size) // stride + 2
        self.patch_proj = nn.Linear(patch_size, d_model)
        self.pos_embed  = nn.Embedding(self.N, d_model)
        self.dropout    = nn.Dropout(0.1)
    def get_raw_patches(self, x):
        B = x.shape[0]
        pad = x[:, -1:].expand(B, self.S)
        x_pad = torch.cat([x, pad], dim=1)
        return x_pad.unfold(1, self.P, self.S)
    def forward(self, x):
        patches  = self.get_raw_patches(x)
        content  = self.patch_proj(patches)
        pos_vecs = self.pos_embed(torch.arange(self.N, device=x.device))
        return self.dropout(content + pos_vecs)

class ConformerFFN(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.W1, self.V, self.W2 = (nn.Linear(d_model, d_ff), nn.Linear(d_model, d_ff),
                                     nn.Linear(d_ff, d_model))
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        x = self.norm(x)
        return self.drop(self.W2(F.silu(self.W1(x)) * self.V(x)))

class ConformerConvModule(nn.Module):
    def __init__(self, d_model, kernel_size=31, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.pw1  = nn.Conv1d(d_model, 2*d_model, 1)
        self.glu  = nn.GLU(dim=1)
        self.dw   = nn.Conv1d(d_model, d_model, kernel_size, padding=kernel_size//2, groups=d_model)
        self.bn   = nn.BatchNorm1d(d_model)
        self.act  = nn.SiLU()
        self.pw2  = nn.Conv1d(d_model, d_model, 1)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        r = x
        x = self.norm(x).transpose(1,2)
        x = self.glu(self.pw1(x))
        x = self.act(self.bn(self.dw(x)))
        x = self.drop(self.pw2(x)).transpose(1,2)
        return r + x

class ConformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout, kernel_size=31):
        super().__init__()
        self.ffn1 = ConformerFFN(d_model, d_ff, dropout)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(d_model)
        self.conv = ConformerConvModule(d_model, kernel_size, dropout)
        self.ffn2 = ConformerFFN(d_model, d_ff, dropout)
        self.norm = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        x = x + 0.5 * self.ffn1(x)
        xn = self.attn_norm(x)
        x  = x + self.drop(self.attn(xn, xn, xn)[0])
        x  = self.conv(x)
        x  = x + 0.5 * self.ffn2(x)
        return self.norm(x)

class TransformerBackbone(nn.Module):
    def __init__(self, d_model, n_heads, n_layers, d_ff, dropout):
        super().__init__()
        self.layers = nn.ModuleList([
            ConformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x):
        for l in self.layers: x = l(x)
        return self.norm(x)

class ReconstructionHead(nn.Module):
    def __init__(self, d_model, patch_size):
        super().__init__()
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model),
                                   nn.GELU(), nn.Linear(d_model, patch_size))
    def forward(self, z): return self.head(z)

class ClassificationHead(nn.Module):
    def __init__(self, d_model, n_classes, dropout=0.1, hidden_dim=None):
        super().__init__()
        if hidden_dim is None: hidden_dim = d_model // 2
        self.attn_pool = nn.Linear(d_model, 1)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, n_classes))
    def forward(self, z):
        w = F.softmax(self.attn_pool(z), dim=1)
        return self.head((w * z).sum(dim=1))

class DenoisingHead(nn.Module):
    def __init__(self, d_model, patch_size, n_patches, stride, spectrum_length):
        super().__init__()
        self.P, self.S, self.N, self.L = patch_size, stride, n_patches, spectrum_length
        self.proj = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model),
                                   nn.GELU(), nn.Linear(d_model, patch_size))
    def forward(self, z):
        B = z.shape[0]
        pr = self.proj(z)
        out = torch.zeros(B, self.L+self.S, device=z.device)
        cnt = torch.zeros(self.L+self.S, device=z.device)
        for k in range(self.N):
            s = k * self.S
            out[:, s:s+self.P] += pr[:, k, :]
            cnt[s:s+self.P]    += 1
        return (out / cnt.clamp(min=1))[:, :self.L]

print('✓ Architecture définie')

✓ Architecture définie


In [15]:
class PatchTSTSSL(nn.Module):
    def __init__(self, patch_embed, backbone, recon_head, mask_ratio=0.4):
        super().__init__()
        self.patch_embed = patch_embed
        self.backbone    = backbone
        self.recon_head  = recon_head
        self.mask_ratio  = mask_ratio
        self.mask_token  = nn.Parameter(torch.zeros(1, 1, patch_embed.D))
        nn.init.trunc_normal_(self.mask_token, std=0.02)
    def forward(self, x):
        B = x.shape[0]
        N = self.patch_embed.N
        patches_orig = self.patch_embed.get_raw_patches(x)
        tokens = self.patch_embed(x)
        n_masked = int(N * self.mask_ratio)
        mask = torch.zeros(B, N, dtype=torch.bool, device=x.device)
        for b in range(B):
            max_start = max(1, N - n_masked)
            start = torch.randint(0, max_start, (1,), device=x.device).item()
            mask[b, start:start+n_masked] = True
        pos_vecs = self.patch_embed.pos_embed(torch.arange(N, device=x.device))
        mask_tok_pos = (self.mask_token.to(x.device) + pos_vecs.unsqueeze(0)).expand(B, -1, -1)
        tokens = torch.where(mask.unsqueeze(-1), mask_tok_pos, tokens)
        z = self.backbone(tokens)
        z_masked      = z[mask]
        patches_recon = self.recon_head(z_masked)
        patches_target= patches_orig[mask]
        loss = F.mse_loss(patches_recon, patches_target)
        return loss, mask

class PatchTSTMultiTask(nn.Module):
    def __init__(self, patch_embed, backbone, class_head, denoise_head, alpha=1.0, beta=0.5):
        super().__init__()
        self.patch_embed, self.backbone = patch_embed, backbone
        self.class_head, self.denoise_head = class_head, denoise_head
        self.alpha, self.beta = alpha, beta
    def encode(self, x):
        return self.backbone(self.patch_embed(x))
    def classify(self, x):
        return self.class_head(self.encode(x))
    def forward(self, x, y=None, clean_target=None):
        z = self.encode(x)
        logits   = self.class_head(z)
        denoised = self.denoise_head(z)
        loss = None
        if y is not None and clean_target is not None:
            loss_clf = F.cross_entropy(logits, y, label_smoothing=0.1)
            loss_den = F.mse_loss(denoised, clean_target)
            loss = self.alpha * loss_clf + self.beta * loss_den
        return logits, denoised, loss

patch_embed  = PatchEmbedding(CFG['L'], CFG['patch_size'], CFG['stride'], CFG['d_model']).to(DEVICE)
backbone     = TransformerBackbone(CFG['d_model'], CFG['n_heads'], CFG['n_layers'],
                                    CFG['d_ff'], CFG['dropout']).to(DEVICE)
recon_head   = ReconstructionHead(CFG['d_model'], CFG['patch_size']).to(DEVICE)
class_head   = ClassificationHead(CFG['d_model'], CFG['N_CLASSES'], dropout=0.1).to(DEVICE)
denoise_head = DenoisingHead(CFG['d_model'], CFG['patch_size'], N_PATCHES,
                              CFG['stride'], CFG['L']).to(DEVICE)

total = sum(p.numel() for m in [patch_embed, backbone, class_head, denoise_head]
            for p in m.parameters() if p.requires_grad)
print(f'Total paramètres : {total:,}')

Total paramètres : 6,596,191


---
## 🛠️ Section 3 — Fonctions d'entraînement


In [16]:
def ssl_train_epoch(model, loader, optimizer, scheduler=None):
    model.train()
    total_loss, n = 0.0, 0
    for x in loader:
        x = x.to(DEVICE)
        loss, _ = model(x)
        optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        if scheduler: scheduler.step()
        total_loss += loss.item() * len(x); n += len(x)
    return total_loss / n

@torch.no_grad()
def ssl_eval_epoch(model, loader):
    model.eval()
    total_loss, n = 0.0, 0
    for x in loader:
        x = x.to(DEVICE)
        loss, _ = model(x)
        total_loss += loss.item() * len(x); n += len(x)
    return total_loss / n

def snr_db(clean, signal):
    noise_power  = ((signal - clean) ** 2).mean(dim=-1) + 1e-8
    signal_power = (clean ** 2).mean(dim=-1) + 1e-8
    return 10 * torch.log10(signal_power / noise_power)

def multitask_train_epoch(model, loader, optimizer):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    total_mse, total_snr = 0.0, 0.0
    for x, x_clean, y in loader:
        x, x_clean, y = x.to(DEVICE), x_clean.to(DEVICE), y.to(DEVICE)
        logits, denoised, loss = model(x, y=y, clean_target=x_clean)
        optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        with torch.no_grad():
            mse = F.mse_loss(denoised, x_clean)
            snr_gain = (snr_db(x_clean, denoised) - snr_db(x_clean, x)).mean()
        total_loss += loss.item()*len(y); correct += (logits.argmax(1)==y).sum().item()
        total_mse += mse.item()*len(y); total_snr += snr_gain.item()*len(y); n += len(y)
    return total_loss/n, correct/n, total_mse/n, total_snr/n

@torch.no_grad()
def multitask_eval_epoch(model, loader):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    total_mse, total_snr = 0.0, 0.0
    for x, x_clean, y in loader:
        x, x_clean, y = x.to(DEVICE), x_clean.to(DEVICE), y.to(DEVICE)
        logits, denoised, loss = model(x, y=y, clean_target=x_clean)
        mse = F.mse_loss(denoised, x_clean)
        snr_gain = (snr_db(x_clean, denoised) - snr_db(x_clean, x)).mean()
        total_loss += loss.item()*len(y); correct += (logits.argmax(1)==y).sum().item()
        total_mse += mse.item()*len(y); total_snr += snr_gain.item()*len(y); n += len(y)
    return total_loss/n, correct/n, total_mse/n, total_snr/n

print('✓ Fonctions définies')

✓ Fonctions définies


---
## 🧠 Phase 1 — SSL Pre-training (CSV uniquement)


In [17]:
ssl_model = PatchTSTSSL(patch_embed, backbone, recon_head, CFG['mask_ratio']).to(DEVICE)
patch_embed.dropout.p = 0.0

n_ssl = len(ssl_dataset)
n_val_ssl = max(1, int(0.2 * n_ssl))
ssl_train_sub, ssl_val_sub = torch.utils.data.random_split(
    ssl_dataset, [n_ssl-n_val_ssl, n_val_ssl],
    generator=torch.Generator().manual_seed(SEED))
ssl_train_loader = DataLoader(ssl_train_sub, batch_size=CFG['ssl_batch'], shuffle=True, num_workers=0)
ssl_val_loader   = DataLoader(ssl_val_sub, batch_size=CFG['ssl_batch'], shuffle=False, num_workers=0)

ssl_optimizer = AdamW(ssl_model.parameters(), lr=CFG['ssl_lr'], weight_decay=1e-4)
ssl_scheduler = OneCycleLR(ssl_optimizer, max_lr=CFG['ssl_lr'], epochs=CFG['ssl_epochs'],
                           steps_per_epoch=len(ssl_train_loader),
                           pct_start=0.1, div_factor=25.0, final_div_factor=1e4)

best_ssl_loss = float('inf')
print(f'=== Phase 1 : SSL Pre-training (CSV seul, {CFG["ssl_epochs"]} époques) ===')
for epoch in range(1, CFG['ssl_epochs']+1):
    tr = ssl_train_epoch(ssl_model, ssl_train_loader, ssl_optimizer, ssl_scheduler)
    va = ssl_eval_epoch(ssl_model, ssl_val_loader)
    safe_log(writer, 'Phase1_SSL/Loss', {'Train': tr, 'Val': va}, epoch, method='add_scalars')
    if va < best_ssl_loss: best_ssl_loss = va
    if epoch % 20 == 0 or epoch == 1:
        print(f'  epoch {epoch:3d} : train={tr:.6f}  val={va:.6f}')

patch_embed.dropout.p = 0.1
torch.save({'patch_embed': patch_embed.state_dict(), 'backbone': backbone.state_dict(),
            'cfg': CFG, 'ssl_val_loss': best_ssl_loss}, CFG['ssl_path'])
print(f'\n✓ Meilleure val MSE : {best_ssl_loss:.6f}')
print(f'✓ Sauvegardé → {CFG["ssl_path"]}')

=== Phase 1 : SSL Pre-training (CSV seul, 100 époques) ===
  epoch   1 : train=0.659397  val=0.522384
  epoch  20 : train=0.020404  val=0.018294
  epoch  40 : train=0.007751  val=0.007730
  epoch  60 : train=0.004145  val=0.004422
  epoch  80 : train=0.003098  val=0.003735
  epoch 100 : train=0.002719  val=0.003741

✓ Meilleure val MSE : 0.003321
✓ Sauvegardé → /home/glider/models/ssl_backbone_csv_withclean.pth


---
## 🔍 Phase 2 — Linear Probing


In [18]:
checkpoint = torch.load(CFG['ssl_path'], map_location=DEVICE, weights_only=False)
patch_embed.load_state_dict(checkpoint['patch_embed'])
backbone.load_state_dict(checkpoint['backbone'])

class_head   = ClassificationHead(CFG['d_model'], CFG['N_CLASSES'], dropout=0.1).to(DEVICE)
denoise_head = DenoisingHead(CFG['d_model'], CFG['patch_size'], N_PATCHES,
                              CFG['stride'], CFG['L']).to(DEVICE)
for p in patch_embed.parameters(): p.requires_grad = False
for p in backbone.parameters():    p.requires_grad = False

probe_model = PatchTSTMultiTask(patch_embed, backbone, class_head, denoise_head,
                                 alpha=CFG['alpha'], beta=CFG['beta']).to(DEVICE)
probe_optimizer = AdamW(filter(lambda p: p.requires_grad, probe_model.parameters()),
                        lr=CFG['probe_lr'], weight_decay=1e-4)

best_probe_acc = 0.0
print(f'=== Phase 2 : Linear Probing ({CFG["probe_epochs"]} époques) ===')
for epoch in range(1, CFG['probe_epochs']+1):
    tr_loss, tr_acc, tr_mse, tr_snr = multitask_train_epoch(probe_model, train_loader, probe_optimizer)
    va_loss, va_acc, va_mse, va_snr = multitask_eval_epoch(probe_model, val_loader)

    safe_log(writer, 'Phase2_Probe/Accuracy', {'Train': tr_acc, 'Val': va_acc}, epoch, method='add_scalars')

    if va_acc > best_probe_acc:
        best_probe_acc = va_acc
        torch.save(probe_model.state_dict(), CFG['probe_path'])

    if epoch % 20 == 0 or epoch == 1:
        print(f'  epoch {epoch:3d} : train={tr_acc:.2%}  val={va_acc:.2%}')

print(f'\n✓ Meilleure Val Acc Phase 2 : {best_probe_acc:.2%}')
print(f'✓ Sauvegardé → {CFG["probe_path"]}')

=== Phase 2 : Linear Probing (80 époques) ===
  epoch   1 : train=44.74%  val=76.44%
  epoch  20 : train=90.56%  val=96.49%
  epoch  40 : train=93.26%  val=96.74%
  epoch  60 : train=95.15%  val=97.49%
  epoch  80 : train=95.12%  val=96.99%

✓ Meilleure Val Acc Phase 2 : 98.25%
✓ Sauvegardé → /home/glider/models/probe_model_csv_withclean.pth


---
## 🎯 Phase 3 — Fine-tuning complet


In [19]:
probe_model.load_state_dict(torch.load(CFG['probe_path'], map_location=DEVICE, weights_only=False))
for p in probe_model.parameters(): p.requires_grad = True

ft_optimizer = AdamW([
    {'params': probe_model.patch_embed.parameters(),  'lr': CFG['ft_lr']},
    {'params': probe_model.backbone.parameters(),     'lr': CFG['ft_lr']},
    {'params': probe_model.class_head.parameters(),   'lr': CFG['ft_lr']*10},
    {'params': probe_model.denoise_head.parameters(), 'lr': CFG['ft_lr']*10},
], weight_decay=1e-4)
ft_scheduler = CosineAnnealingLR(ft_optimizer, T_max=CFG['ft_epochs'], eta_min=1e-6)

best_ft_acc = 0.0
print(f'=== Phase 3 : Fine-tuning ({CFG["ft_epochs"]} époques) ===')
for epoch in range(1, CFG['ft_epochs']+1):
    tr_loss, tr_acc, tr_mse, tr_snr = multitask_train_epoch(probe_model, train_loader, ft_optimizer)
    va_loss, va_acc, va_mse, va_snr = multitask_eval_epoch(probe_model, val_loader)
    ft_scheduler.step()

    safe_log(writer, 'Phase3_FT/Accuracy', {'Train': tr_acc, 'Val': va_acc}, epoch, method='add_scalars')

    if epoch % 5 == 0:
        torch.save({'model_state': probe_model.state_dict(), 'epoch': epoch, 'cfg': CFG},
                   os.path.join(HOME, 'models', 'checkpoint_csv_withclean_latest.pth'))

    if va_acc > best_ft_acc:
        best_ft_acc = va_acc
        torch.save({'model_state': probe_model.state_dict(), 'cfg': CFG,
                    'le_classes': le.classes_, 'val_acc': best_ft_acc}, CFG['final_path'])

    if epoch % 10 == 0 or epoch == 1:
        print(f'  epoch {epoch:3d} : train={tr_acc:.2%}  val={va_acc:.2%}')

writer.flush()
print(f'\n✓ Meilleure Val Acc Phase 3 : {best_ft_acc:.2%}')
print(f'✓ Sauvegardé → {CFG["final_path"]}')
print('🔴 TÉLÉCHARGE final_model_csv_withclean.pth vers ton Mac')

=== Phase 3 : Fine-tuning (80 époques) ===
  epoch   1 : train=95.52%  val=96.74%
  epoch  10 : train=97.16%  val=96.99%
  epoch  20 : train=97.16%  val=99.00%
  epoch  30 : train=98.03%  val=98.75%
  epoch  40 : train=98.29%  val=98.25%
  epoch  50 : train=98.03%  val=99.00%
  epoch  60 : train=98.58%  val=99.00%
  epoch  70 : train=98.29%  val=99.00%
  epoch  80 : train=98.65%  val=99.00%

✓ Meilleure Val Acc Phase 3 : 99.25%
✓ Sauvegardé → /home/glider/models/final_model_csv_withclean.pth
🔴 TÉLÉCHARGE final_model_csv_withclean.pth vers ton Mac


---
## 📊 Évaluation finale — comparaison Run A vs Run B


In [20]:
ckpt = torch.load(CFG['final_path'], map_location=DEVICE, weights_only=False)
probe_model.load_state_dict(ckpt['model_state'])
probe_model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for x, x_clean, y in test_loader:
        logits, _, _ = probe_model(x.to(DEVICE))
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(y.numpy())

final_acc = accuracy_score(all_labels, all_preds)

print('═'*65)
print('  RUN B — CSV bruités + CSV propres ajoutés au train (SANS .npy)')
print('═'*65)
print(f'  Train : {train_dataset.n_noisy} bruités + {train_dataset.n_clean} propres = {len(train_dataset)}')
print(f'  Test  : {len(test_dataset)} (bruités uniquement, FIXE)')
print(f'  Test Accuracy : {final_acc:.2%}')
print('═'*65)
print()
print('  Comparaison avec le Run A (CSV bruités seuls, propres = proxy uniquement) :')
print('    Run A (config différente, référence historique) : 94.21%')
print(f'    Run B (ce notebook, mêmes bruités + propres AJOUTÉS) : {final_acc:.2%}')

═════════════════════════════════════════════════════════════════
  RUN B — CSV bruités + CSV propres ajoutés au train (SANS .npy)
═════════════════════════════════════════════════════════════════
  Train : 1864 bruités + 881 propres = 2745
  Test  : 400 (bruités uniquement, FIXE)
  Test Accuracy : 98.75%
═════════════════════════════════════════════════════════════════

  Comparaison avec le Run A (CSV bruités seuls, propres = proxy uniquement) :
    Run A (config différente, référence historique) : 94.21%
    Run B (ce notebook, mêmes bruités + propres AJOUTÉS) : 98.75%


In [21]:
print(classification_report(all_labels, all_preds, target_names=le.classes_, zero_division=0))

                  precision    recall  f1-score   support

             ABS       1.00      1.00      1.00        21
         ACRYLIC       1.00      1.00      1.00        11
       CELLULOSE       1.00      1.00      1.00        11
        CHITOSAN       1.00      1.00      1.00        19
             ENR       1.00      1.00      1.00        13
            EPDM       1.00      1.00      1.00        11
             EVA       1.00      1.00      1.00        19
            HDPE       1.00      1.00      1.00         9
            LDPE       1.00      1.00      1.00        10
           NYLON       1.00      1.00      1.00         9
            PBAT       1.00      0.90      0.95        10
             PBS       1.00      0.83      0.91        12
              PC       1.00      1.00      1.00        11
            PEEK       1.00      1.00      1.00        10
             PEI       1.00      1.00      1.00        10
             PET       1.00      1.00      1.00        13
PF THERMOPLAS

In [22]:
# ── Logger dans le dashboard du point 2 ────────────────────────────────────
from torch.utils.tensorboard import SummaryWriter as SW2

HPARAMS_POINT2 = os.path.join(HOME, 'runs', 'hparams_point2_npy_importance')

def log_run(cfg, metrics, run_label):
    run_dir = os.path.join(HPARAMS_POINT2, run_label)
    w = SW2(run_dir)
    hparams_clean = {k: v for k, v in cfg.items() if isinstance(v, (int, float, str, bool))}
    w.add_hparams(hparams_clean, metrics)
    w.close()
    print(f'✓ Run "{run_label}" loggé')

log_run(
    cfg={'strategy': 'csv_with_clean_added', 'n_noisy_train': train_dataset.n_noisy,
         'n_clean_added': train_dataset.n_clean, 'uses_npy': False},
    metrics={'test_acc': final_acc},
    run_label='run_B_csv_with_clean',
)
print(f'\nDashboard : tensorboard --logdir {HPARAMS_POINT2} --port 6009')

✓ Run "run_B_csv_with_clean" loggé

Dashboard : tensorboard --logdir /home/glider/runs/hparams_point2_npy_importance --port 6009


In [24]:
# ── Test supplémentaire : le modèle Run B généralise-t-il vers .npy ? ─────
NPY_ROOT  = os.path.join(HOME, 'data', '2026-FTIR-Preprocesed','2026 - FTIR - 4. Selected Datasets - Preprocessed')
TEST_DIR  = os.path.join(NPY_ROOT, '1.2 TestSet - UptoY dB')

npy_test_clean = np.load(os.path.join(TEST_DIR, 'TestGroundTruthSet_Pre.npy'))
npy_test_noisy = np.load(os.path.join(TEST_DIR, 'TestNoisySet_Upto30SNR_Pre.npy'))

N_PER_CLASS_NPY_TEST = npy_test_clean.shape[0] // N_CLASSES
npy_labels_test = np.repeat(np.arange(N_CLASSES), N_PER_CLASS_NPY_TEST)

npy_test_dataset = SimpleMultiTaskDataset(npy_test_noisy, npy_test_clean, npy_labels_test)
npy_test_loader  = DataLoader(npy_test_dataset, batch_size=CFG['batch_size'], shuffle=False, num_workers=0)

probe_model.eval()
all_preds_npy, all_labels_npy = [], []
with torch.no_grad():
    for x, x_clean, y in npy_test_loader:
        logits, _, _ = probe_model(x.to(DEVICE))
        all_preds_npy.extend(logits.argmax(1).cpu().numpy())
        all_labels_npy.extend(y.numpy())

acc_npy_from_runB = accuracy_score(all_labels_npy, all_preds_npy)

print('═'*65)
print('  RUN B testé sur .npy (jamais vu pendant l\'entraînement)')
print('═'*65)
print(f'  Test .npy : {acc_npy_from_runB:.2%}')
print(f'  (Comparaison — modèle .npy seul → test .npy : 95.25%)')
print(f'  (Comparaison — modèle .npy seul → test CSV  : 23.47%, l\'autre sens du gap)')
print('═'*65)

═════════════════════════════════════════════════════════════════
  RUN B testé sur .npy (jamais vu pendant l'entraînement)
═════════════════════════════════════════════════════════════════
  Test .npy : 7.27%
  (Comparaison — modèle .npy seul → test .npy : 95.25%)
  (Comparaison — modèle .npy seul → test CSV  : 23.47%, l'autre sens du gap)
═════════════════════════════════════════════════════════════════
